# Task 2: Forecasting Models

This notebook implements a chronological train/test split, ARIMA, SARIMA, LSTM, and model comparison using MAE, RMSE, and MAPE.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import DATA_DIR, FIGURES_DIR, REPORTS_DIR, TEST_START_DATE
from src.data_loader import fetch_all_assets, pivot_adjusted_close
from src.preprocessing import clean_all_assets
from src.modeling import fit_arima_forecast, fit_sarima_forecast, fit_lstm_forecast
from src.visualization import plot_model_forecasts

In [ ]:
prices_path = DATA_DIR / "adjusted_close_prices.csv"
if prices_path.exists():
    prices = pd.read_csv(prices_path, index_col="Date", parse_dates=True)
else:
    prices = pivot_adjusted_close(clean_all_assets(fetch_all_assets()))
    prices.to_csv(prices_path)

tsla = prices["TSLA"].dropna()
tsla.tail()

## Chronological Split Rationale
Random shuffling would leak future market information into training. The split uses observations before 2025-01-01 for training and 2025 onward for testing.

## ARIMA Baseline
The documented baseline order is ARIMA(1,1,1). The differencing term d=1 is consistent with the typical non-stationarity of price levels.

In [ ]:
arima_results, train, test, arima_forecast, arima_metrics = fit_arima_forecast(tsla, split_date=TEST_START_DATE, order=(1,1,1))
arima_metrics.as_dict()

## SARIMA Baseline
The documented SARIMA configuration is SARIMA(1,1,1)x(1,0,1,5). The seasonal period m=5 represents a weekly trading-day pattern.

In [ ]:
sarima_results, _, _, sarima_forecast, sarima_metrics = fit_sarima_forecast(tsla, split_date=TEST_START_DATE, order=(1,1,1), seasonal_order=(1,0,1,5))
sarima_metrics.as_dict()

## LSTM Model
The LSTM uses 60-day windowed sequence data, MinMax scaling, two LSTM layers, dropout, and dense output layers. This cell may take longer because TensorFlow trains a neural network.

In [ ]:
# Uncomment to train LSTM in the notebook environment.
# lstm_model, _, _, lstm_forecast, lstm_metrics = fit_lstm_forecast(tsla, split_date=TEST_START_DATE, window_size=60, epochs=10, batch_size=32)
# lstm_metrics.as_dict()

## Model Comparison

In [ ]:
metrics = [arima_metrics.as_dict(), sarima_metrics.as_dict()]
forecast_dict = {"ARIMA(1,1,1)": arima_forecast, "SARIMA(1,1,1)x(1,0,1,5)": sarima_forecast}
# If LSTM was trained above, add it:
# metrics.append(lstm_metrics.as_dict())
# forecast_dict["LSTM window=60"] = lstm_forecast
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(REPORTS_DIR / "task2_model_comparison.csv", index=False)
plot_model_forecasts(test, forecast_dict, FIGURES_DIR / "task2_forecast_comparison.png");
metrics_df